# K-Sentinel: real MRI acquisition variations and natural failures

Author: Vinsy Moyya, ORCID 0009-0003-0042-3556. An exploratory extension of [K-Sentinel](https://github.com/vinsymoyya/k-sentinel).

Uses original-creators' M4Raw MRI acquisitions (Lyu et al., Scientific Data 10:264, DOI 10.1038/s41597-023-02181-4), licensed CC BY 4.0, pinned source revision. Code MIT. The older synthetic dataset remains CC0; M4Raw is separately attributed and licensed.

100 disjoint participants; 180 source acquisitions; actual repeated scans and T2/FLAIR contrast; participant-level motion-risk stress cohort. Retrospective masks are simulated acquisition changes. No artificial structures, synthetic motion or synthetic noise are added. All-available source references contain noise and possible motion. This experiment does not validate clinical anatomy, diagnosis or prospective scanning benefit.


In [ ]:
from pathlib import Path
import os, sys, zipfile, subprocess, json
archive = next(Path('/kaggle/input').rglob('k-sentinel-0.3.0-source.bundle'))
destination = Path('/kaggle/working')
with zipfile.ZipFile(archive) as z:
    for member in z.namelist():
        if not (destination/member).resolve().is_relative_to(destination.resolve()):
            raise ValueError('Unsafe source member')
    z.extractall(destination)
root = destination/'k-sentinel-0.3.0'
os.chdir(root); sys.path.insert(0,str(root))
import numpy as np, torch, h5py, requests
print({'numpy':np.__version__,'torch':str(torch.__version__),'h5py':h5py.__version__})


## Validate both workflows
The existing 45 cases remain separate from the new real-acquisition contracts. No skips count as passes.

In [ ]:
subprocess.run([sys.executable,'scripts/run_validation.py','--output','/kaggle/working/validation/legacy'],check=True)
subprocess.run([sys.executable,'scripts/run_real_validation.py','--output','/kaggle/working/validation/real'],check=True)


## Fetch the frozen, byte-verified real acquisition selection
About 2.22 GB, 180 files. Every source SHA256 is checked. Internet is required for this step. All selection and settings are recorded in research/real-acquisition/PROTOCOL.md before model evaluation.

In [ ]:
subprocess.run([sys.executable,'scripts/download_m4raw.py','--output','/kaggle/temp/m4raw','--workers','4'],check=True)


## Train and evaluate
Eight fixed fine-tuning epochs on twenty training participants. Thirty separate calibration participants. Twenty-five expanded-test participants and twenty-five motion-risk participants. Three fixed native slices per acquisition, all four coils. Test and calibration data never select weights.

In [ ]:
subprocess.run([sys.executable,'scripts/run_real_acquisition.py','--data','/kaggle/temp/m4raw','--output','/kaggle/working/real-results'],check=True)


## Inspect every outcome
A negative audit is not an anatomical certificate. Motion-subset membership is not a file-level artifact label. Natural failures here are reference-reconstruction errors; no disease or focal-lesion labels are supplied.

In [ ]:
report = json.loads(Path('/kaggle/working/real-results/summary.json').read_text())
primary = [g for g in report['groups'] if g['role']=='test' and g['sequence']=='T2' and g['repetition']==1 and g['pattern']=='random' and g['acceleration']==4]
for group in primary:
    print({key:group[key] for key in ('method','participants','mean_rss_nrmse','flagged','high_reference_error_cases','unflagged_high_reference_error')})
print('Paired real repetitions:',report['paired_repetition_error_change'])
print('Worst unflagged cases:',[(r['scan_id'],r['rss_nrmse']) for r in report['worst_unflagged_cases']])


## Reproduce and review
Download all result rows, source checks, checkpoint and configuration from output. Preserve the participant groups and original dataset attribution. Review statistical calibration, source-reference limitations, native failure cases and actual versus retrospective acquisition changes. This is a single-scanner, healthy-volunteer exploratory study.